# Experimentations — les 4 modeles sur nos vraies images

Ce notebook est dedie **uniquement a l'EXPERIMENTATION**.
Les cas de tests du prof sont dans l'autre notebook : **`models_cas_test.ipynb`**.

## Ce qu'on cherche a montrer

1. **Sous-apprentissage vs sur-apprentissage** — la pathologie propre a chaque modele
2. **L'effet de chaque hyperparametre**, modele par modele, courbe a l'appui
3. **Une comparaison finale** des 4 (precision, ecart train/test, duree)

## Les donnees

Nos **750 photos** (250 par classe : aucun / humain / animal), redimensionnees en
**64x64x3 = 12288 entrees**, decoupees en **600 entrainement / 150 test** (80/20 **stratifie**,
donc 50 images de chaque classe dans le test).

## Methode

> **Une seule verite chiffree dans tout le projet.**
> Ce notebook utilise **exactement les hyperparametres de production** (ceux des scripts
> `python/train_*.py`). Chaque courbe fait varier **UN SEUL** hyperparametre et garde
> **tous les autres a leur valeur de production**. Le point correspondant a la configuration
> de production redonne donc **le meme chiffre que `train_*.py`** — aucune contradiction
> possible entre le notebook, les scripts et le rapport.

Sur chaque graphe, la **ligne grise en pointilles** est la **baseline** (le score d'un modele
qui repond toujours la meme classe) et la **ligne verte** marque **notre choix de production**.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), "..", "python"))

import numpy as np
import matplotlib.pyplot as plt
from bindings import MLP, ModeleLineaire, SVM, RBFNetwork, UnContreTous

np.random.seed(0)
CLASSES = ["aucun", "humain", "animal"]


def charger(n):
    return np.load(os.path.join("..", "datasets", n))


X_train, y_train = charger("X_train.npy"), charger("y_train.npy")
X_test,  y_test  = charger("X_test.npy"),  charger("y_test.npy")
N_ENTREES = X_train.shape[1]

# La BASELINE : score d'un modele qui repond toujours la classe la plus frequente.
# C'est LE repere : tout ce qui est en dessous n'a rien appris.
vals, cnt = np.unique(y_test, return_counts=True)
BASELINE = 100.0 * cnt.max() / len(y_test)
etat = "equilibre" if cnt.max() == cnt.min() else "desequilibre"

print(f"Train : {len(X_train)} images | Test : {len(X_test)} images | {N_ENTREES} entrees")
print(f"Test {etat} : {'/'.join(str(c) for c in cnt)}")
print(f"BASELINE (repond toujours la meme classe) : {BASELINE:.1f}%  <- le score a battre")

# ---------------------------------------------------------------------------
#  CONFIGURATION DE PRODUCTION : identique a python/train_*.py.
#  Chaque courbe fait varier UN SEUL parametre et garde les autres a ces valeurs
#  -> le point "production" redonne exactement le chiffre des scripts train_*.py.
# ---------------------------------------------------------------------------
PROD = {
    "mlp": {"cachee": 32, "steps": 50_000, "lr": 0.01, "activation": "tanh"},
    "lin": {"lr": 0.01, "epochs": 50},
    "svm": {"lr": 0.001, "epochs": 300, "gamma": 0.001, "max_ex": 60},
    "rbf": {"k": 20, "gamma": 0.01, "iters": 15},
}
print(f"\nConfiguration de production (= python/train_*.py) :")
for m, p in PROD.items():
    print(f"  {m:4s} : {p}")


def one_hot_pm1(y, nc=3):
    """Etiquette 2 -> [-1, -1, +1]  (codage +/-1, adapte a tanh)."""
    Y = -np.ones((len(y), nc)); Y[np.arange(len(y)), y] = 1.0
    return Y


def precision(f, X, y):
    """f(x) -> liste de scores (un par classe) ; la classe predite = argmax."""
    return 100.0 * np.mean([int(np.argmax(f(x)) == int(v)) for x, v in zip(X.tolist(), y)])


# --- Fabriques : construisent + entrainent un modele a partir d'une config ----------

def fab_mlp(cachee=None, steps=None, lr=None, activation=None, X=None, y=None):
    p = PROD["mlp"]
    cachee = p["cachee"] if cachee is None else cachee
    steps = p["steps"] if steps is None else steps
    lr = p["lr"] if lr is None else lr
    activation = p["activation"] if activation is None else activation
    X = X_train if X is None else X; y = y_train if y is None else y
    m = MLP([N_ENTREES, cachee, len(CLASSES)], activation=activation)
    m.fit(X, one_hot_pm1(y), steps=steps, lr=lr, is_classification=True)
    return lambda x, m=m: m.predict(list(x))


def fab_lin(lr=None, epochs=None):
    p = PROD["lin"]
    lr = p["lr"] if lr is None else lr
    epochs = p["epochs"] if epochs is None else epochs
    u = UnContreTous.entrainer(ModeleLineaire, X_train.tolist(), y_train.tolist(),
                               len(CLASSES), lr=lr, epochs=epochs)
    return lambda x, u=u: u.predict(list(x))


def sous_echantillon(n):
    """Le SVM garde TOUS ses exemples (cout O(n^2)) -> on limite. Tirage reproductible."""
    i = np.random.RandomState(0).permutation(len(X_train))[:n]
    return X_train[i], y_train[i]


def fab_svm(lr=None, epochs=None, gamma=None, max_ex=None):
    p = PROD["svm"]
    lr = p["lr"] if lr is None else lr
    epochs = p["epochs"] if epochs is None else epochs
    gamma = p["gamma"] if gamma is None else gamma
    max_ex = p["max_ex"] if max_ex is None else max_ex
    Xs, ys = sous_echantillon(max_ex)
    u = UnContreTous.entrainer(SVM, Xs.tolist(), ys.tolist(), len(CLASSES),
                               lr=lr, epochs=epochs, gamma=gamma)
    return (lambda x, u=u: u.predict(list(x))), Xs, ys


def fab_rbf(k=None, gamma=None, iters=None):
    p = PROD["rbf"]
    k = p["k"] if k is None else k
    gamma = p["gamma"] if gamma is None else gamma
    iters = p["iters"] if iters is None else iters
    rs = []
    for c in range(len(CLASSES)):
        cible = np.where(y_train == c, 1.0, -1.0)          # un-contre-tous
        r = RBFNetwork(k=k, gamma=gamma)
        r.train(X_train.tolist(), cible.tolist(), k=k, iterations=iters)
        rs.append(r)
    return lambda x, rs=rs: [r.predict(list(x)) for r in rs]


def trace(x, courbes, titre, xlabel, prod=None, ylabel="precision (%)", logx=False):
    """Courbes + baseline (gris) + notre choix de production (vert)."""
    plt.figure(figsize=(8, 4.6))
    for nom, ys in courbes.items():
        plt.plot(x, ys, "o-", label=nom)
    plt.axhline(BASELINE, color="gray", ls="--", label=f"baseline ({BASELINE:.1f}%)")
    if prod is not None:
        plt.axvline(prod, color="green", ls=":", lw=2, label=f"notre choix ({prod})")
    if logx:
        plt.xscale("log")
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(titre)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


print("\nPret. /!\\ Ce notebook entraine une centaine de modeles : comptez ~40 min.")

Train : 600 images | Test : 150 images | 12288 entrees
Test equilibre : 50/50/50
BASELINE (repond toujours la meme classe) : 33.3%  <- le score a battre

Configuration de production (= python/train_*.py) :
  mlp  : {'cachee': 32, 'steps': 50000, 'lr': 0.01, 'activation': 'tanh'}
  lin  : {'lr': 0.01, 'epochs': 50}
  svm  : {'lr': 0.001, 'epochs': 300, 'gamma': 0.001, 'max_ex': 60}
  rbf  : {'k': 20, 'gamma': 0.01, 'iters': 15}

Pret. /!\ Ce notebook entraine une centaine de modeles : comptez ~40 min.


## 1. Sous-apprentissage vs sur-apprentissage

**Le concept central du cours.** On compare la precision sur l'**entrainement** et sur le **test** :

| Signature | Diagnostic |
|---|---|
| train ≈ test ≈ baseline | **sous-apprentissage** — le modele est trop faible, il n'apprend rien |
| train >> test | **sur-apprentissage** — il apprend par coeur, il ne generalise pas |
| train ≈ test, nettement > baseline | **bon equilibre** |

> Les 4 modeles sont entraines ici avec **la configuration de production** :
> ces chiffres sont donc **exactement** ceux qu'affichent les scripts `python/train_*.py`.

In [ ]:
resultats = {}

# --- MLP (multi-classe natif : 3 neurones de sortie)
t0 = time.perf_counter(); f_mlp = fab_mlp(); t = time.perf_counter() - t0
resultats["MLP"] = (precision(f_mlp, X_train, y_train), precision(f_mlp, X_test, y_test), t)
print(f"MLP      : {resultats['MLP'][0]:.1f}% / {resultats['MLP'][1]:.1f}%  ({t:.1f}s)")

# --- Lineaire (binaire -> un-contre-tous)
t0 = time.perf_counter(); f_lin = fab_lin(); t = time.perf_counter() - t0
resultats["Lineaire"] = (precision(f_lin, X_train, y_train), precision(f_lin, X_test, y_test), t)
print(f"Lineaire : {resultats['Lineaire'][0]:.1f}% / {resultats['Lineaire'][1]:.1f}%  ({t:.1f}s)")

# --- SVM (binaire -> un-contre-tous, sur un sous-echantillon)
t0 = time.perf_counter(); f_svm, Xs, ys = fab_svm(); t = time.perf_counter() - t0
# /!\ la precision d'ENTRAINEMENT du SVM se mesure sur les exemples qu'il a VUS (les 60)
resultats["SVM"] = (precision(f_svm, Xs, ys), precision(f_svm, X_test, y_test), t)
print(f"SVM      : {resultats['SVM'][0]:.1f}% / {resultats['SVM'][1]:.1f}%  ({t:.1f}s)")

# --- RBF (un-contre-tous : 1 reseau par classe)
t0 = time.perf_counter(); f_rbf = fab_rbf(); t = time.perf_counter() - t0
resultats["RBF"] = (precision(f_rbf, X_train, y_train), precision(f_rbf, X_test, y_test), t)
print(f"RBF      : {resultats['RBF'][0]:.1f}% / {resultats['RBF'][1]:.1f}%  ({t:.1f}s)")

# --- Graphe
noms = list(resultats)
tr = [resultats[n][0] for n in noms]; te = [resultats[n][1] for n in noms]
pos = np.arange(len(noms))
plt.figure(figsize=(8.5, 4.8))
plt.bar(pos - 0.2, tr, 0.4, label="entrainement")
plt.bar(pos + 0.2, te, 0.4, label="test")
plt.axhline(BASELINE, color="gray", ls="--", lw=2, label=f"baseline ({BASELINE:.1f}%)")
plt.xticks(pos, noms); plt.ylabel("precision (%)"); plt.ylim(0, 105)
plt.title("Sous-apprentissage vs sur-apprentissage : entrainement VS test")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

print(f"\n{'Modele':10s} {'train':>8s} {'test':>8s} {'ecart':>9s} {'vs base':>9s} {'duree':>8s}   diagnostic")
print("-" * 82)
for n in noms:
    a, b, t = resultats[n]
    ecart = a - b
    if ecart > 20:            diag = "SUR-apprentissage (apprend par coeur)"
    elif b <= BASELINE + 2:   diag = "SOUS-apprentissage (n'apprend rien)"
    else:                     diag = "bon equilibre"
    print(f"{n:10s} {a:7.1f}% {b:7.1f}% {ecart:8.1f}pt {b-BASELINE:+8.1f}pt {t:7.1f}s   {diag}")

## 2. MLP — hyperparametres (Nina)

Les 4 leviers : **nombre de neurones caches**, **learning rate**, **nombre d'iterations**,
**fonction d'activation**.

### 2.1 Nombre de neurones caches

Plus de neurones = plus de capacite. On cherche le point ou le modele est assez riche
sans sur-apprendre.

In [ ]:
neurones = [2, 8, 16, 32, 64]
tr, te = [], []
for h in neurones:
    f = fab_mlp(cachee=h)
    tr.append(precision(f, X_train, y_train)); te.append(precision(f, X_test, y_test))
    print(f"  {h:3d} neurones -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%")

trace(neurones, {"entrainement": tr, "test": te},
      "MLP : effet du nombre de neurones caches", "neurones caches",
      prod=PROD["mlp"]["cachee"])

### 2.2 Learning rate

Trop grand -> le modele **oscille / diverge**. Trop petit -> il **converge trop lentement**.

In [ ]:
lrs = [0.0001, 0.001, 0.01, 0.1, 0.5]
te = []
for lr in lrs:
    f = fab_mlp(lr=lr)
    te.append(precision(f, X_test, y_test))
    print(f"  lr={lr:<8} -> test {te[-1]:5.1f}%")

trace(lrs, {"test": te}, "MLP : effet du learning rate",
      "learning rate (echelle log)", prod=PROD["mlp"]["lr"], logx=True)

### 2.3 Courbe d'apprentissage

Est-ce que le modele **progresse** au fil des iterations ? Une courbe **plate** signifie
qu'il **n'apprend pas** — ici, a cause de la **saturation de tanh** sur 12288 entrees.

In [ ]:
paliers = [1_000, 5_000, 10_000, 20_000, 50_000]
tr, te = [], []
for s in paliers:
    f = fab_mlp(steps=s)
    tr.append(precision(f, X_train, y_train)); te.append(precision(f, X_test, y_test))
    print(f"  {s:6d} iterations -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%")

trace(paliers, {"entrainement": tr, "test": te},
      "MLP : courbe d'apprentissage", "nombre d'iterations", prod=PROD["mlp"]["steps"])

### 2.4 Fonction d'activation

**Rappel important** : l'activation choisie ne s'applique qu'aux **couches CACHEES**.
La couche de sortie reste **tanh** en classification (sortie dans [-1,1], compatible avec
nos etiquettes +/-1) ou **lineaire** en regression.

In [ ]:
activs = ["tanh", "sigmoid", "relu"]
tr, te = [], []
for a in activs:
    f = fab_mlp(activation=a)
    tr.append(precision(f, X_train, y_train)); te.append(precision(f, X_test, y_test))
    print(f"  {a:8s} -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%")

pos = np.arange(len(activs))
plt.figure(figsize=(7.5, 4.4))
plt.bar(pos - 0.2, tr, 0.4, label="entrainement")
plt.bar(pos + 0.2, te, 0.4, label="test")
plt.axhline(BASELINE, color="gray", ls="--", lw=2, label=f"baseline ({BASELINE:.1f}%)")
plt.xticks(pos, activs); plt.ylabel("precision (%)")
plt.title("MLP : activation des couches cachees (la sortie reste tanh)")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

## 3. Modele lineaire — hyperparametres (Valentin)

Le perceptron n'a que **2 leviers** : le **learning rate** et le **nombre d'epochs**.
C'est le modele le plus simple du projet... et le plus limite.

In [ ]:
epochs = [5, 20, 50, 100, 200]
te = []
for e in epochs:
    f = fab_lin(epochs=e)
    te.append(precision(f, X_test, y_test))
    print(f"  {e:4d} epochs -> test {te[-1]:5.1f}%")

trace(epochs, {"test": te}, "Lineaire : effet du nombre d'epochs",
      "epochs", prod=PROD["lin"]["epochs"])

In [ ]:
lrs = [0.0001, 0.001, 0.01, 0.1]
te = []
for lr in lrs:
    f = fab_lin(lr=lr)
    te.append(precision(f, X_test, y_test))
    print(f"  lr={lr:<8} -> test {te[-1]:5.1f}%")

trace(lrs, {"test": te}, "Lineaire : effet du learning rate",
      "learning rate (echelle log)", prod=PROD["lin"]["lr"], logx=True)

## 4. SVM — hyperparametres (Valentin)

### 4.1 Gamma (largeur du noyau)

- `gamma = 0` -> noyau **lineaire** (produit scalaire), non borne -> peut **diverger**
- `gamma > 0` -> noyau **RBF** : `K(a,b) = exp(-gamma * ||a-b||^2)`, toujours dans [0, 1]
- gamma **trop grand** -> chaque point ne "voit" que lui-meme -> **sur-apprentissage**
- gamma **trop petit** -> tous les points se ressemblent -> **sous-apprentissage**

L'**ecart train/test** sur ce graphe mesure directement le sur-apprentissage.

In [ ]:
gammas = [0.0001, 0.001, 0.01, 0.1]
tr, te = [], []
for g in gammas:
    f, Xg, yg = fab_svm(gamma=g)
    tr.append(precision(f, Xg, yg)); te.append(precision(f, X_test, y_test))
    print(f"  gamma={g:<8} -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%  (ecart {tr[-1]-te[-1]:5.1f}pt)")

trace(gammas, {"entrainement": tr, "test": te},
      "SVM : effet de gamma  (l'ecart train/test = sur-apprentissage)",
      "gamma (echelle log)", prod=PROD["svm"]["gamma"], logx=True)

### 4.2 Nombre d'exemples — **la justification de notre `MAX_EXEMPLES = 60`**

Le SVM a noyau **conserve TOUS ses exemples d'entrainement** :

`f(x) = biais + somme_n( alpha_n * y_n * K(x_n, x) )`

-> memoire en **O(n x d)** et entrainement en **O(n^2)**. Sur des images a 12288 dimensions,
le modele devient vite ingerable (le JSON depassait 40 Mo). D'ou notre sous-echantillonnage.

**Ce graphe mesure le compromis precision / temps qui justifie ce choix.**

In [ ]:
tailles = [20, 40, 60, 100, 150]
te, temps = [], []
for n in tailles:
    t0 = time.perf_counter()
    f, _, _ = fab_svm(max_ex=n)
    temps.append(time.perf_counter() - t0)
    te.append(precision(f, X_test, y_test))
    print(f"  {n:4d} exemples -> test {te[-1]:5.1f}%   entrainement {temps[-1]:7.1f}s")

fig, ax1 = plt.subplots(figsize=(8.5, 4.8))
ax1.plot(tailles, te, "o-", color="tab:blue", label="precision test")
ax1.axhline(BASELINE, color="gray", ls="--", label=f"baseline ({BASELINE:.1f}%)")
ax1.axvline(PROD["svm"]["max_ex"], color="green", ls=":", lw=2,
            label=f"notre choix ({PROD['svm']['max_ex']})")
ax1.set_xlabel("nombre d'exemples d'entrainement")
ax1.set_ylabel("precision test (%)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(tailles, temps, "s--", color="tab:red")
ax2.set_ylabel("temps d'entrainement (s)", color="tab:red")
plt.title("SVM : precision et temps selon le nombre d'exemples (cout en O(n^2))")
ax1.legend(loc="center right"); plt.tight_layout(); plt.show()

## 5. RBF — hyperparametres (Ali)

Le RBF **compresse** les 12288 dimensions en **k distances a k centres** (trouves par k-means).
C'est une forme de **reduction de dimension** — et c'est ce qui explique ses resultats ici.

### 5.1 Nombre de centres (k)

`k` est la **capacite** du modele :
- k **trop petit** -> pas assez de prototypes -> **sous-apprentissage**
- k **trop grand** -> il memorise -> **sur-apprentissage**

In [ ]:
ks = [5, 10, 20, 40, 60]
tr, te = [], []
for k in ks:
    f = fab_rbf(k=k)
    tr.append(precision(f, X_train, y_train)); te.append(precision(f, X_test, y_test))
    print(f"  k={k:3d} centres -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%  (ecart {tr[-1]-te[-1]:5.1f}pt)")

trace(ks, {"entrainement": tr, "test": te},
      "RBF : effet du nombre de centres k", "k (nombre de centres)", prod=PROD["rbf"]["k"])

### 5.2 Gamma (largeur des gaussiennes)

In [ ]:
gammas = [0.0001, 0.001, 0.01, 0.1, 1.0]
tr, te = [], []
for g in gammas:
    f = fab_rbf(gamma=g)
    tr.append(precision(f, X_train, y_train)); te.append(precision(f, X_test, y_test))
    print(f"  gamma={g:<8} -> train {tr[-1]:5.1f}%  test {te[-1]:5.1f}%")

trace(gammas, {"entrainement": tr, "test": te},
      "RBF : effet de gamma", "gamma (echelle log)", prod=PROD["rbf"]["gamma"], logx=True)

## 6. Comparaison finale des 4 modeles

Configuration de production pour tous. **Le gain vs baseline est la seule mesure qui compte** :
un modele qui ne la depasse pas n'a rien appris.

In [ ]:
import pandas as pd

lignes = []
for n, (a, b, t) in resultats.items():
    ecart = a - b
    if ecart > 20:            diag = "sur-apprentissage"
    elif b <= BASELINE + 2:   diag = "sous-apprentissage"
    else:                     diag = "bon equilibre"
    lignes.append({"Modele": n, "Train (%)": round(a, 1), "Test (%)": round(b, 1),
                   "Ecart (pt)": round(ecart, 1),
                   "Gain vs baseline (pt)": round(b - BASELINE, 1),
                   "Duree (s)": round(t, 1), "Diagnostic": diag})

df = pd.DataFrame(lignes).sort_values("Test (%)", ascending=False).set_index("Modele")
print(f"Baseline = {BASELINE:.1f}%  (un modele qui n'apprend rien)")
df

### Precision par classe

Revele si un modele **s'effondre** sur une seule classe (il repond toujours pareil)
au lieu d'apprendre reellement a discriminer.

In [ ]:
fs = {"MLP": f_mlp, "Lineaire": f_lin, "SVM": f_svm, "RBF": f_rbf}
largeur = 0.2
pos = np.arange(len(CLASSES))
plt.figure(figsize=(9.5, 4.8))
for k, (nom, f) in enumerate(fs.items()):
    accs = [precision(f, X_test[np.where(y_test == c)[0]], y_test[np.where(y_test == c)[0]])
            for c in range(len(CLASSES))]
    plt.bar(pos + (k - 1.5) * largeur, accs, largeur, label=nom)
    print(f"  {nom:9s} : " + "  ".join(f"{CLASSES[c]}={accs[c]:5.1f}%" for c in range(len(CLASSES))))
plt.axhline(BASELINE, color="gray", ls="--", lw=2, label=f"baseline ({BASELINE:.1f}%)")
plt.xticks(pos, CLASSES); plt.ylabel("precision (%)")
plt.title("Precision par classe : quel modele s'effondre sur une seule classe ?")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

## Conclusion

**A relire et completer avec les chiffres obtenus ci-dessus.**

1. **L'equilibrage du dataset a change l'interpretation.** Avec 230/81/38 images, la baseline
   etait a **66.7%** et **aucun** modele ne la battait : ils exploitaient simplement le
   desequilibre en repondant toujours "aucun". Avec **250/250/250**, la baseline tombe a
   **33.3%** et les resultats deviennent enfin **interpretables**.

2. **Les deux pathologies du cours sont presentes, chacune sur un modele different** :
   le **SVM sur-apprend** (ecart train/test enorme : il memorise ses 60 exemples),
   le **MLP et le lineaire sous-apprennent** (train ≈ test ≈ baseline).

3. **Le MLP sature.** Sur 12288 entrees, la somme ponderee est trop grande -> tanh sort ≈ ±1
   -> sa derivee (1 - x²) tend vers **0** -> **gradient nul** -> plus d'apprentissage.
   **La preuve** : d'une execution a l'autre, il s'effondre sur une classe **differente**
   (choisie par le hasard de l'initialisation, pas par les donnees).
   **Piste identifiee, non implementee** : initialisation **Xavier** (poids / sqrt(nb entrees)).

4. **Le RBF generalise le mieux** : ses `k` centres **compressent** 12288 dimensions en
   k distances -> il contourne le fleau de la dimension que le MLP subit de plein fouet.

5. **Limite du dataset lui-meme** : la classe `humain` est la plus difficile — nos photos
   sont des scenes de rue ou l'humain occupe peu de pixels ; apres reduction en 64x64,
   l'information disparait. C'est une limite des **donnees**, pas seulement des modeles.